# 第 9 章：正交性、Gram-Schmidt 與 QR 分解

本 Notebook 與 `ch09_orthogonality_qr.md` 教學文件對照，示範：

1. 正交向量的判斷
2. 正交矩陣的性質（$Q^TQ=QQ^T=I$，保長度變換）
3. 手刻 Gram-Schmidt 正交化流程（含逐步輸出）
4. 用 `np.linalg.qr` 驗證手刻結果一致
5. 驗證 $QR \approx A$ 與 $Q^TQ \approx I$

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

## 1. 正交向量的判斷

兩向量正交 $\iff$ 內積為 0。

In [2]:
u = np.array([1, 0])
v = np.array([0, 1])
print("u =", u, ", v =", v)
print("u . v =", np.dot(u, v), "-> 正交" if np.isclose(np.dot(u, v), 0) else "-> 不正交")

a = np.array([1, 2])
b = np.array([3, -1])
print("\na =", a, ", b =", b)
print("a . b =", np.dot(a, b), "-> 正交" if np.isclose(np.dot(a, b), 0) else "-> 不正交")

u = [1 0] , v = [0 1]
u . v = 0 -> 正交

a = [1 2] , b = [ 3 -1]
a . b = 1 -> 不正交


## 2. 正交矩陣的性質

正交矩陣 $Q$ 滿足 $Q^TQ = QQ^T = I$，且對任意向量 $x$，$\|Qx\| = \|x\|$（保長度變換）。以旋轉矩陣為例：

In [3]:
theta = np.pi / 3  # 60 度旋轉矩陣
Q_rot = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)],
])
print("旋轉矩陣 Q (theta=60°):\n", Q_rot)
print("Q^T Q:\n", Q_rot.T @ Q_rot)
print("Q^T Q ≈ I ?", np.allclose(Q_rot.T @ Q_rot, np.eye(2)))

x = np.array([3.0, 4.0])
Qx = Q_rot @ x
print("\n原向量 x =", x, ", |x| =", np.linalg.norm(x))
print("變換後 Qx =", Qx, ", |Qx| =", np.linalg.norm(Qx))
print("長度是否保持不變 ?", np.isclose(np.linalg.norm(x), np.linalg.norm(Qx)))

旋轉矩陣 Q (theta=60°):
 [[ 0.5   -0.866]
 [ 0.866  0.5  ]]
Q^T Q:
 [[ 1. -0.]
 [-0.  1.]]
Q^T Q ≈ I ? True

原向量 x = [3. 4.] , |x| = 5.0
變換後 Qx = [-1.9641  4.5981] , |Qx| = 5.0
長度是否保持不變 ? True


## 3. 手刻 Gram-Schmidt 正交化流程

對應教學文件中的手算範例：

$$
a_1 = (1,1,0), \quad a_2 = (2,0,2), \quad a_3 = (3,3,3)
$$

Gram-Schmidt 遞迴公式：$u_k = a_k - \sum_{i=1}^{k-1} \mathrm{proj}_{u_i}(a_k)$，再標準化 $e_i = u_i / \|u_i\|$。

In [4]:
def gram_schmidt(A, verbose=False):
    """對矩陣 A（行向量為欲正交化的向量組）做 Gram-Schmidt 正交化與標準化。

    回傳 Q（標準正交基底，shape (m, n)）、R（上三角矩陣，shape (n, n)），滿足 A = Q @ R。
    """
    m, n = A.shape
    U = np.zeros((m, n))
    Q = np.zeros((m, n))
    R = np.zeros((n, n))

    for j in range(n):
        a_j = A[:, j].astype(float).copy()
        u_j = a_j.copy()
        for i in range(j):
            coeff = Q[:, i] @ a_j
            R[i, j] = coeff
            u_j -= coeff * Q[:, i]
            if verbose:
                print(f"  a_{j+1} 在 e_{i+1} 方向上的投影係數 = {coeff:.4f}")
        norm_u = np.linalg.norm(u_j)
        R[j, j] = norm_u
        U[:, j] = u_j
        Q[:, j] = u_j / norm_u
        if verbose:
            print(f"  u_{j+1} (尚未標準化) = {u_j}")
            print(f"  ||u_{j+1}|| = {norm_u:.4f}")
            print(f"  e_{j+1} (標準化後) = {Q[:, j]}")
            print()

    return Q, R, U

In [5]:
a1 = np.array([1, 1, 0])
a2 = np.array([2, 0, 2])
a3 = np.array([3, 3, 3])
A = np.column_stack([a1, a2, a3]).astype(float)

print("矩陣 A = [a1 a2 a3]:\n", A)
print()

Q_manual, R_manual, U_manual = gram_schmidt(A, verbose=True)

print("正交化後（尚未標準化）的向量:")
print("u1 =", U_manual[:, 0])
print("u2 =", U_manual[:, 1])
print("u3 =", U_manual[:, 2])

print("\n驗證正交性（內積應為 0）:")
print("u1 . u2 =", np.dot(U_manual[:, 0], U_manual[:, 1]))
print("u1 . u3 =", np.dot(U_manual[:, 0], U_manual[:, 2]))
print("u2 . u3 =", np.dot(U_manual[:, 1], U_manual[:, 2]))

矩陣 A = [a1 a2 a3]:
 [[1. 2. 3.]
 [1. 0. 3.]
 [0. 2. 3.]]

  u_1 (尚未標準化) = [1. 1. 0.]
  ||u_1|| = 1.4142
  e_1 (標準化後) = [0.7071 0.7071 0.    ]

  a_2 在 e_1 方向上的投影係數 = 1.4142
  u_2 (尚未標準化) = [ 1. -1.  2.]
  ||u_2|| = 2.4495
  e_2 (標準化後) = [ 0.4082 -0.4082  0.8165]

  a_3 在 e_1 方向上的投影係數 = 4.2426
  a_3 在 e_2 方向上的投影係數 = 2.4495
  u_3 (尚未標準化) = [-1.  1.  1.]
  ||u_3|| = 1.7321
  e_3 (標準化後) = [-0.5774  0.5774  0.5774]

正交化後（尚未標準化）的向量:
u1 = [1. 1. 0.]
u2 = [ 1. -1.  2.]
u3 = [-1.  1.  1.]

驗證正交性（內積應為 0）:
u1 . u2 = 4.440892098500626e-16
u1 . u3 = 6.661338147750939e-16
u2 . u3 = -2.4424906541753444e-15


In [6]:
print("標準化後的標準正交基底 Q（各行向量）:")
print("e1 =", Q_manual[:, 0], ", ||e1|| =", np.linalg.norm(Q_manual[:, 0]))
print("e2 =", Q_manual[:, 1], ", ||e2|| =", np.linalg.norm(Q_manual[:, 1]))
print("e3 =", Q_manual[:, 2], ", ||e3|| =", np.linalg.norm(Q_manual[:, 2]))

print("\n手刻 Gram-Schmidt 得到的 R (上三角矩陣):\n", R_manual)

標準化後的標準正交基底 Q（各行向量）:
e1 = [0.7071 0.7071 0.    ] , ||e1|| = 0.9999999999999999
e2 = [ 0.4082 -0.4082  0.8165] , ||e2|| = 1.0
e3 = [-0.5774  0.5774  0.5774] , ||e3|| = 1.0

手刻 Gram-Schmidt 得到的 R (上三角矩陣):
 [[1.4142 1.4142 4.2426]
 [0.     2.4495 2.4495]
 [0.     0.     1.7321]]


## 4. 用 `np.linalg.qr` 驗證手刻結果

NumPy 內建的 `np.linalg.qr` 使用數值上更穩定的演算法（非古典 Gram-Schmidt），但結果應與手刻版本一致（正交基底可能差一個正負號，這是正常現象，因為正交基底不唯一）。

In [7]:
Q_np, R_np = np.linalg.qr(A)

print("np.linalg.qr 得到的 Q:\n", Q_np)
print("\nnp.linalg.qr 得到的 R:\n", R_np)

same_up_to_sign = np.allclose(np.abs(Q_manual), np.abs(Q_np))
print("\n手刻 Q 與 np.linalg.qr 的 Q 是否只差正負號:", same_up_to_sign)

print("\n手刻 Q 的行向量兩兩正交 (Q_manual^T @ Q_manual ≈ I) ?",
      np.allclose(Q_manual.T @ Q_manual, np.eye(3)))
print("np.linalg.qr 的 Q 的行向量兩兩正交 (Q_np^T @ Q_np ≈ I) ?",
      np.allclose(Q_np.T @ Q_np, np.eye(3)))

np.linalg.qr 得到的 Q:
 [[-0.7071  0.4082 -0.5774]
 [-0.7071 -0.4082  0.5774]
 [-0.      0.8165  0.5774]]

np.linalg.qr 得到的 R:
 [[-1.4142 -1.4142 -4.2426]
 [ 0.      2.4495  2.4495]
 [ 0.      0.      1.7321]]

手刻 Q 與 np.linalg.qr 的 Q 是否只差正負號: True

手刻 Q 的行向量兩兩正交 (Q_manual^T @ Q_manual ≈ I) ? True
np.linalg.qr 的 Q 的行向量兩兩正交 (Q_np^T @ Q_np ≈ I) ? True


## 5. 最終驗證：$QR \approx A$ 與 $Q^TQ \approx I$

In [8]:
print("手刻版本: Q_manual @ R_manual ≈ A ?", np.allclose(Q_manual @ R_manual, A))
print("手刻版本: Q_manual.T @ Q_manual ≈ I ?", np.allclose(Q_manual.T @ Q_manual, np.eye(3)))

print("\nnp.linalg.qr 版本: Q_np @ R_np ≈ A ?", np.allclose(Q_np @ R_np, A))
print("np.linalg.qr 版本: Q_np.T @ Q_np ≈ I ?", np.allclose(Q_np.T @ Q_np, np.eye(3)))

all_checks_passed = (
    np.allclose(Q_manual @ R_manual, A)
    and np.allclose(Q_manual.T @ Q_manual, np.eye(3))
    and np.allclose(Q_np @ R_np, A)
    and np.allclose(Q_np.T @ Q_np, np.eye(3))
    and same_up_to_sign
)

print()
print("所有驗證通過！" if all_checks_passed else "警告：部分驗證未通過。")

手刻版本: Q_manual @ R_manual ≈ A ? True
手刻版本: Q_manual.T @ Q_manual ≈ I ? True

np.linalg.qr 版本: Q_np @ R_np ≈ A ? True
np.linalg.qr 版本: Q_np.T @ Q_np ≈ I ? True

所有驗證通過！


## 重點整理

- 兩向量正交 $\iff$ 內積為 0。
- 正交矩陣 $Q$ 滿足 $Q^TQ=QQ^T=I$，保持向量長度與內積（幾何上為旋轉或鏡射）。
- Gram-Schmidt：依序將新向量減去在先前正交向量上的投影，再標準化即得標準正交基底。
- QR 分解 $A=QR$：$Q$ 為標準正交基底組成的矩陣，$R$ 為上三角矩陣，兩者與 Gram-Schmidt 過程直接對應。
- `np.linalg.qr` 與手刻 Gram-Schmidt 給出等價（僅差正負號）的結果，且皆滿足 $QR=A$、$Q^TQ=I$。